# Neural Networks
Now we get to the main work of this lecture, designing your first neural network. For this we will be using a basic toy problem which is useful for building intuition on how neural networks function.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import Module
import torch.optim as optim
from torch.optim import Optimizer
from torch.utils.data import Dataset, DataLoader, random_split
from torch import Tensor
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision.transforms as transforms
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import time
import pandas as pd
type LossFN = Union[Module]
# Find device being used
device = (
    "cuda" # For NVIDIA GPUs
    if torch.cuda.is_available()
    else "mps" # For Apple devies
    if torch.backends.mps.is_available()
    else "cpu"
)


## XOR Problem
The XOR problem is a toy problem where our goal is to learn XOR. But first what is XOR? Exclusive Or (XOR) in programming is a logic gate found within many computers. In Python we can call it with `^`, for example `0 ^ 1` will be `1` in Python. For the purposes of this task it is just a function (or rule) that we want to be able to find with our neural network. Below is a truth table for XOR, this will be our dataset.

| X | Y | X^Y |
| - | - | --- |
| 0 | 0 |  0  |
| 0 | 1 |  1  |
| 1 | 0 |  1  |
| 1 | 1 |  0  |

We've skipped the work of creating the dataset and given it to you below. Keep in mind you normally would split this dataset into a training and test set however since its so small we skipped this part.

In [ ]:
# This a is common toy example within neural networks
# In this dictionary the key (bool, bool) is the data,
# and the item bool is the label. (False, False) has label False, etc.
xor_data = {(False, False): False, (False, True): True, (True, False): True, (True, True): False}

class XORDataset(Dataset):
    def __init__(self, data):
        """Initiliase the XOR problem's dataset"""
        # This gets the keys from the dataset
        inputs = [list(k) for k in data.keys()]
        labels = [v for v in data.values()]

        # Like before we make sure we are using tensors as the output
        self.input = torch.tensor(inputs, dtype=torch.float32)
        self.label = torch.tensor(labels, dtype=torch.float32).view(-1, 1)

    def __len__(self) -> int:
        """Get length of our dataset"""
        # Simply return length of our data
        return len(self.input)

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        """Get an item from idx"""
        x = self.input[idx] # get data from inputs
        y = self.label[idx] # get data from label
        return x, y # return them as a tuple so we can unpack

# Wrap the classes in a dataloader, this will allow us to easily retrieve the data
# In this example since we only have 4 bits of data we don't split the data.
xor_dataset = XORDataset(xor_data)
xor_dataloader = DataLoader(xor_dataset, batch_size=1, shuffle=True)

# Print dataset
for batch_idx, (x, y) in enumerate(xor_dataloader):
    print(f"Batch {batch_idx + 1}\nInputs: {x}\nLabels: {y}")

## Modules (Neural Networks)
In PyTorch all neural networks are represented through the `Module` abstract base class. The idea behind this is that you are able to construct neural networks by linking modules (which themselves are neural networks) together to create larger neural networks.

Modules require two functions to be implemented.
- `__init__` which stores the layers of the network and its activation functions.
- `forward` which takes the input tensor and does the forward pass by running its layers and associated activation functions as to turn the input tensor into a prediction. Each layer is required to start and end with the same amount of neurons as the last.

Below is an example of the model for our XOR problem. To construct a solution to this problem we use one hidden layer and one output layer. Each of these layers using the sigmoid function to introduce non-linearity into the dataset. To keep things basic we use the `nn.Linear` module to create basic neural network layers.

In [ ]:
# Define a simple feedforward neural network
class XORNetwork(Module):
    def __init__(self):
        """Use this to store layers and activation functions"""
        super().__init__() # This is required as it calls its parents
        # Layers
        self.hidden = nn.Linear(2, 2) # Takes two inputs returns two
        self.output = nn.Linear(2, 1) # Takes two as layer above is the same.
        # Activation Function
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x: Tensor):
        """Run the forward pass with this data"""
        # Forward pass: input -> hidden layer -> sigmoid -> output layer -> sigmoid
        x = self.sigmoid(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

### Training & Testing
Once we have created our model we can start to train it and testing it. Before we start this process there are 3 things that need to be done, first create our model by calling its class, then define a loss function. Think back to linear regression at any given point we first need to calculate how well we are doing at any given step. This starts with our loss function. The last step is defining our optimiser. This is once again similar to linear regression, but rather than creating our own version we can use PyTorch's. Once we have done these three things we can continue.

The training stage follows the steps:
1. Set model to `model.train()`, this ensures any changes to the weights and biases are done.
2. Iterate through batches within the data loader.
3. Assign the iterated values to a device `X, y = X.to(device), y.to(device)`. This is important if we want our model to train fast.
4. Compute the predicted error (*forward pass*). This is done by running the model `model(X)`, before calculating the loss. Think linear regression.
5. Run backpropogation (*back pass*). This is done by calculating the gradients for each neuron. Once we have these gradients we can change the weights and biases by stepping the optimiser. After this we ensure all gradients are set to zero for the next iteration.
6. Optionally print the loss.

Below is an example of this on our XOR problem.

In [ ]:
# Create network
model = XORNetwork().to(device)

# Define the loss function and the optimizer
loss_fn = nn.BCELoss()  # Binary Cross Entropy Loss
optimiser = optim.SGD(model.parameters(), lr=0.1)

# Training loop
epochs = 10000
model.train()
for epoch in range(epochs):

    for X, y in xor_dataloader:
        # Forward pass
        y_pred = model(X)
        
        # Compute the loss
        loss = loss_fn(y_pred, y)
        
        # Backward pass and optimization
        loss.backward() # Calculate gradients
        optimiser.step() # Change weights and biases as a result of gradients
        optimiser.zero_grad() # Set gradients to zero
        
    # Print loss every 1000 epochs
    if (epoch+1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')

# Test the model, this is usually done multiple times
model.eval()
with torch.no_grad():  # No need to track gradients for testing
    inputs = torch.tensor([0.0, 0.0])
    test_output = model(inputs)
    print(f"\nGiven input: 0, 0\nResult: {test_output[0]}")

The above results should should end up converging as training goes on with the example test of the input (0, 0) producing a value that is close to 0. In general neural networks function on their ability to generalise so in many cases even with thousands on epochs the result should never be 0.

### Modules as building blocks
To further reinforce the idea that each layer is just a building block we can easily reuse the module we just made by simply putting it in our new network. For example below we expand our network by adding an extra layer. This ability to compose networks in this way allows use to improve architectures made by others. It also means that `Linear`, and `Sigmoid` are also both modules that can be used. Although in our case here introducing extra layers doesn't really increase our accuracy in anyway.

In [ ]:
class BigXORNetwork(Module):
    def __init__(self):
        """Use this to store layers and activation functions"""
        super().__init__() # This is required
        self.hidden = nn.Linear(2, 2)
        self.output = XORNetwork() # We can reuse this here
        # Activation Function
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x: Tensor):
        """Run the forward pass with this data"""
        # Forward pass: input -> hidden layer -> hidden layer -> sigmoid -> output layer -> sigmoid
        x = self.sigmoid(self.hidden(x))
        x = self.output(x)
        return x

## Three-Bit Adder
From the above content you should now have a general idea on the entire process of building a neural network, from creating a model to training/testing. Now its up to you to implement a similar example. For this problem we introduce our new problem, a three-bit adder. This takes in two three-digit binary numbers and computes their result outputting each of the five bits. Below are some basic examples of addition.
```
  010      110      101
+ 100    + 111    + 111
 0110     1101     1100
```

Our goal is to construct a model similar to the XOR model that is able to use our new dataset. We have provided you with a dictionary of the data but the rest is up to you.

In [ ]:
from itertools import product

def three_bit_adder(a2, a1, a0, b2, b1, b0):
    """Returns the 4 bits of sum: (s3, s2, s1, s0)"""
    a = (a2 << 2) | (a1 << 1) | a0
    b = (b2 << 2) | (b1 << 1) | b0
    sum_ = a + b
    s3 = (sum_ >> 3) & 1
    s2 = (sum_ >> 2) & 1
    s1 = (sum_ >> 1) & 1
    s0 = sum_ & 1
    return (s3, s2, s1, s0)

# Generate all 64 input combinations (3+3 bits)
inputs = list(product([0, 1], repeat=6))
three_bit_add_data = {inp: three_bit_adder(*inp) for inp in inputs}

### **(Question 1)** Three-Bit Adder Dataset
Similar to the previous section your first step is to implement a dataset so that we can actually train our model. Be sure to use the `three_bit_add_data` as a basis for your dataset. 

**Implement a dataset that returns three tuple which include all 3 input bits, and all 4 output bits.** You may have to change some things to get it to work this time.

In [ ]:
class ThreeBitAdditionDataset(Dataset):
    def __init__(self, data):
        """Initialise dataset for the three bit adder"""
        # Your Code Goes Here
        pass
    def __len__(self):
        # Your Code Goes Here
        pass
    def __getitem__(self, idx):
        # Your Code Goes Here
        pass
three_bit_dataset = ThreeBitAdditionDataset(three_bit_add_data)
three_bit_dataloader = DataLoader(three_bit_dataset, batch_size=1, shuffle=True)

### **(Question 2)** Three-Bit Adder Network
The next step is to create our model. This can be done similar to last-time except you will most likely require more layers. Some functions which may be helpful are `nn.ReLu`, `nn.Sequential`, and `nn.Linear`. 

**Implement a model.** Keep in mind until the next section we won't be able to see how well our code works so for now focus on getting something that can be used for the next section.

In [ ]:
class ThreeBitAdderNet(nn.Module):
    def __init__(self):
        """Model layers for a three-bit adder neural network"""
        # Your Code Goes Here
        pass
    def forward(self, x):
        """Forward function to compute the result"""
        # Your Code Goes Here
        pass


### **(Question 3)** Train your network
Using your previous knowledge you should be able to create a loop that is able to train your neural network. Afterwards it is probably best if you can then test its accuracy by using a testing loop (except with the same data of course).

**Implement a training loop.**

In [ ]:
# Your Code Goes Here
pass


After this you should be able to run the bottom code to see how well you are able to compute your result. Look out for any cases that seem to be incorrect. If all have gone well your model should compute all results correctly.

In [ ]:
# A helper function to test the entire network to see if it works...
model.eval()
with torch.no_grad():
    print("\nAll possible 3-bit addition results:")
    for x in product([0.0, 1.0], repeat=6):
        x_tensor = torch.tensor([x], dtype=torch.float32)
        out = model(x_tensor)
        bits = (out[0] > 0.5).int().tolist() # Round to make the check obvious
        print(f"Input: {x}  Output: {bits}")
        assert list(three_bit_add_data[x]) == bits, f"Combination isn't equal: {three_bit_add_data[x]} {bits}"

### Bringing it all together
From this notebook you have put together a basic neural network that is able to simulate functions and rules that you find in the while. This is important as this is the main point of neural networks they are really just a method of approximating a function by using optimisation. The problems we have focused on today are mostly trival as they can already be replicated with easily with our computer's circuits. But where neural networks truly are amazing is in their ability to approximate non-linear rules of many dimensions. Next week we will see this as we develop a neural network for a picture. A type of data which would traditionally be very hard to simulate using a deterministic method.